## LAB PRACTICAL 1
#### IMPLEMENTATION OF CONTINUOS BAG OF WORDS (CBOW) FROM SCRATCH

- Dhruv Yadav
250103007001

In [1]:
import requests
from bs4 import BeautifulSoup
import re
import numpy as np
import time


In [2]:
# web scraping text from URLs to create a massive dataset
def scrape_text_from_url(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text,  "html.parser")

        # Extracting text from all pragraph tags
        paragraphs = soup.find_all('p')
        text = " ".join([p.get_text() for p in paragraphs])

        # Basic cleaning
        text = text.lower()
        text = re.sub(r'\[.*?\]', '', text) # reference brackets "[]"
        text = re.sub(r'[^a-z0-9\s.]', '', text)

        return text
    except requests.exceptions.RequestException as e:
        print(f"Error Scraping {url} : {e}")
        return " "

ai_urls = [
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://en.wikipedia.org/wiki/Artificial_intelligence"

    "https://en.wikipedia.org/wiki/Generative_artificial_intelligence",
    "https://en.wikipedia.org/wiki/History_of_artificial_intelligence",


]

# Inclusion of below URLs would increase the vocab size to 9000+
# "https://en.wikipedia.org/wiki/Deep_learning",
# "https://en.wikipedia.org/wiki/Natural_language_processing",
# "https://en.wikipedia.org/wiki/Computer_vision",
# "https://en.wikipedia.org/wiki/Reinforcement_learning",
# "https://en.wikipedia.org/wiki/Turing_test",
# "https://en.wikipedia.org/wiki/Artificial_neural_network",
# "https://en.wikipedia.org/wiki/Large_language_model"




In [3]:
master_corpus = " "
print("Starting corpus aggregation...")

for url in ai_urls:
    print(f"Fetching: {url}")
    scraped_text = scrape_text_from_url(url)

    if scraped_text:
        master_corpus += scraped_text + " "

    time.sleep(1)

# verify
tokens = master_corpus.split()
vocab = sorted(set(tokens))


print("\n--- Aggregation Complete ---")
print(f"Total Words (Tokens): {len(tokens)}")
print(f"Unique Words (Vocabulary Size): {len(vocab)}")
V = len(vocab)
print("Vocabulary size:", V)


word_to_ix = {w: i for i, w in enumerate(vocab)}
ix_to_word = {i: w for w, i in word_to_ix.items()}



Starting corpus aggregation...
Fetching: https://en.wikipedia.org/wiki/Machine_learning
Fetching: https://en.wikipedia.org/wiki/Artificial_intelligencehttps://en.wikipedia.org/wiki/Generative_artificial_intelligence
Error Scraping https://en.wikipedia.org/wiki/Artificial_intelligencehttps://en.wikipedia.org/wiki/Generative_artificial_intelligence : 403 Client Error: Too many requests. Please respect our robot policy https://w.wiki/4wJS. (dd12474) for url: https://en.wikipedia.org/wiki/Artificial_intelligencehttps://en.wikipedia.org/wiki/Generative_artificial_intelligence
Fetching: https://en.wikipedia.org/wiki/History_of_artificial_intelligence

--- Aggregation Complete ---
Total Words (Tokens): 19218
Unique Words (Vocabulary Size): 4553
Vocabulary size: 4553


In [4]:
## Creating a CBOW dataset

window_size = 2

def make_cbow_data(tokens, window):
    data = []
    for i in range(window, len(tokens) - window):
        context = tokens[i-window:i] + tokens[i+1:i+window+1]
        target = tokens[i]
        data.append((context, target))
    return data

cbow_data = make_cbow_data(tokens, window_size)
print("CBOW samples:", len(cbow_data))
# print(cbow_data)

CBOW samples: 19214


In [5]:
print(cbow_data[:10])
# saving master corpus in a file, cbow data of that corrpus
with open("master_corpus.txt", "w") as file:
    file.write(str(cbow_data))

[(['machine', 'learning', 'is', 'a'], 'ml'), (['learning', 'ml', 'a', 'field'], 'is'), (['ml', 'is', 'field', 'of'], 'a'), (['is', 'a', 'of', 'study'], 'field'), (['a', 'field', 'study', 'in'], 'of'), (['field', 'of', 'in', 'artificial'], 'study'), (['of', 'study', 'artificial', 'intelligence'], 'in'), (['study', 'in', 'intelligence', 'concerned'], 'artificial'), (['in', 'artificial', 'concerned', 'with'], 'intelligence'), (['artificial', 'intelligence', 'with', 'the'], 'concerned')]


In [6]:
# One-Hot encoding
def one_hot(word):
    vec = np.zeros(V)
    vec[word_to_ix[word]] = 1
    return vec

In [7]:
# Initialize Word Vectors

N = 50  # embedding dimension

W1 = np.random.randn(N, V) * 0.01
b1 = np.zeros((N, 1))
W2 = np.random.randn(V, N) * 0.01
b2 = np.zeros((V, 1))

In [8]:
# Activation functions

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def softmax(z):
    exp_z = np.exp(z - np.max(z))
    return exp_z / np.sum(exp_z)

In [9]:

# Training Loop (Forward + Backward Propagation)

lr = 0.05
epochs = 100

for epoch in range(epochs):
    loss_epoch = 0
    for context, target in cbow_data:
        # -------- Forward pass --------
        x = np.zeros((V, 1))
        for w in context:
            x += one_hot(w).reshape(V, 1)
        x /= len(context)

        z1 = W1 @ x + b1
        h = relu(z1)
        z2 = W2 @ h + b2
        y_hat = softmax(z2)

        y = one_hot(target).reshape(V, 1)
        loss = -np.sum(y * np.log(y_hat + 1e-9))
        loss_epoch += loss

        # -------- Backward pass --------
        dz2 = y_hat - y
        dW2 = dz2 @ h.T
        db2 = dz2

        dh = W2.T @ dz2
        dz1 = dh * relu_derivative(z1)
        dW1 = dz1 @ x.T
        db1 = dz1

        # -------- Parameter update --------
        W2 -= lr * dW2
        b2 -= lr * db2
        W1 -= lr * dW1
        b1 -= lr * db1

    print(f"Epoch {epoch+1}, Loss: {loss_epoch/len(cbow_data):.4f}")


Epoch 1, Loss: 6.9830
Epoch 2, Loss: 6.7228
Epoch 3, Loss: 6.5781
Epoch 4, Loss: 6.4365
Epoch 5, Loss: 6.2982
Epoch 6, Loss: 6.1610
Epoch 7, Loss: 6.0331
Epoch 8, Loss: 5.8914
Epoch 9, Loss: 5.7631
Epoch 10, Loss: 5.6431
Epoch 11, Loss: 5.5108
Epoch 12, Loss: 5.3869
Epoch 13, Loss: 5.2659
Epoch 14, Loss: 5.1531
Epoch 15, Loss: 5.0327
Epoch 16, Loss: 4.9184
Epoch 17, Loss: 4.8070
Epoch 18, Loss: 4.6979
Epoch 19, Loss: 4.5956
Epoch 20, Loss: 4.4895
Epoch 21, Loss: 4.3855
Epoch 22, Loss: 4.2831
Epoch 23, Loss: 4.1801
Epoch 24, Loss: 4.0800
Epoch 25, Loss: 3.9810
Epoch 26, Loss: 3.8822
Epoch 27, Loss: 3.7829
Epoch 28, Loss: 3.6853
Epoch 29, Loss: 3.5888
Epoch 30, Loss: 3.4932
Epoch 31, Loss: 3.3977
Epoch 32, Loss: 3.3044
Epoch 33, Loss: 3.2126
Epoch 34, Loss: 3.1200
Epoch 35, Loss: 3.0280
Epoch 36, Loss: 2.9380
Epoch 37, Loss: 2.8484
Epoch 38, Loss: 2.7611
Epoch 39, Loss: 2.6742
Epoch 40, Loss: 2.5873
Epoch 41, Loss: 2.5034
Epoch 42, Loss: 2.4189
Epoch 43, Loss: 2.3369
Epoch 44, Loss: 2.25

In [15]:
# Testing: Predicting Target word


def predict(context):
    x = np.zeros((V, 1))
    for w in context:
        x += one_hot(w).reshape(V, 1)
    x /= len(context)

    h = relu(W1 @ x + b1)
    y_hat = softmax(W2 @ h + b2)
    return ix_to_word[np.argmax(y_hat)]

sample_context, sample_target = cbow_data[300]
print("Context:", sample_context)
print("True target:", sample_target)
print("Predicted:", predict(sample_context))

Context: ['algorithms', 'imitated', 'reasoning', 'humans']
True target: stepbystep
Predicted: symposium


### Methods to increase the accuracy
- increase the dataset size
- stop word removal - captures the context in a better way
- hyperparameter tuning ( here the hyperparams are , embedding dimension (50) and   window_size)
- Instead of using numpy, pytorch can be used!